## VibeVoice Hindi-7B Notebook
This notebook acts as a standalone runner to perform Hindi TTS using the model `sg123321/vibevoice-hindi-7b`. It processes a `.txt` file containing your text and outputs a `.wav` locally to your system.

**Note**: By default, it runs on Colab's Free CPU. However, a 7-Billion parameter language model inference takes significantly long on a CPU. It is **highly recommended** to go to **Runtime > Change runtime type** and select a **T4 GPU**.

In [ ]:
# @title 1. Setup Environment
import os

!git clone https://github.com/vibevoice-community/VibeVoice.git
%cd VibeVoice

!pip install accelerate==1.6.0 transformers==4.51.3 datasets==3.5.0 peft llvmlite>=0.40.0 numba>=0.57.0 diffusers tqdm numpy scipy librosa ml-collections absl-py av aiortc

os.makedirs("demo/voices", exist_ok=True)
!wget -q -O demo/voices/hi-Priya_woman.wav "https://huggingface.co/sg123321/vibevoice-hindi-7b/resolve/main/hi-Priya_woman.wav"

print("Setup Complete!")

In [ ]:
# @title 2. Load the Model
import torch
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

model_path = "sg123321/vibevoice-hindi-7b"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: Running a 7B model on CPU will be extremely slow. We strongly recommend switching to a T4 GPU.\n(Runtime > Change runtime type > T4 GPU)")

load_dtype = torch.bfloat16 if device == "cuda" else torch.float32
attn_impl = "flash_attention_2" if device == "cuda" else "sdpa"

print(f"Loading processor from {model_path}...")
processor = VibeVoiceProcessor.from_pretrained(model_path)

print(f"Loading model (Downloads ~35GB, requires patience)...")
try:
    model = VibeVoiceForConditionalGenerationInference.from_pretrained(
        model_path,
        torch_dtype=load_dtype,
        device_map=device,
        attn_implementation=attn_impl,
    )
except Exception as e:
    print(f"Failed with {attn_impl}, falling back to sdpa...")
    attn_impl = "sdpa"
    model = VibeVoiceForConditionalGenerationInference.from_pretrained(
        model_path,
        torch_dtype=load_dtype,
        device_map=device,
        attn_implementation=attn_impl,
    )

model.eval()
model.set_ddpm_inference_steps(num_steps=10)
print("Model loaded successfully!")

In [ ]:
# @title 3. Upload a Text File
from google.colab import files
import os
import shutil

print("Please upload your text file (e.g. devnagri-test.txt)")
uploaded = files.upload()

uploaded_file_path = None
for filename in uploaded.keys():
    uploaded_file_path = filename
    print(f"File '{filename}' successfully uploaded!")
    break

In [ ]:
# @title 4. Generate TTS and Download 
import time
from google.colab import files

if 'uploaded_file_path' not in locals() or not uploaded_file_path:
    print("Please run the upstream block to upload a text file first.")
else:
    with open(uploaded_file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
        
    lines = content.split('\n')
    if lines:
        lines[0] = f"Speaker 1: {lines[0]}"
    formatted_script = '\n'.join(lines).replace("’", "'") 
    
    voice_path = "demo/voices/hi-Priya_woman.wav"
    
    print("Preparing inputs...")
    inputs = processor(
        text=[formatted_script],
        voice_samples=[[voice_path]],
        padding=True,
        return_tensors="pt",
        return_attention_mask=True,
    )
    
    for k, v in inputs.items():
        if torch.is_tensor(v):
            inputs[k] = v.to(device)
            
    print("Running generation... this will take a while, especially on CPU.")
    start_time = time.time()
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=None,
            cfg_scale=1.3,
            tokenizer=processor.tokenizer,
            generation_config={'do_sample': False},
            verbose=True,
            is_prefill=True,
        )
        
    gen_time = time.time() - start_time
    print(f"Generation took {gen_time:.2f} seconds.")
    
    os.makedirs("outputs", exist_ok=True)
    out_filename = f"outputs/{os.path.splitext(uploaded_file_path)[0]}_generated.wav"
    
    if outputs.speech_outputs and outputs.speech_outputs[0] is not None:
        processor.save_audio(outputs.speech_outputs[0], output_path=out_filename)
        print(f"Audio saved to {out_filename}! Prompting browser download...")
        files.download(out_filename)
    else:
        print("Error: No speech generated.")